In [17]:
import pandas as pd

# Import the CCUS Projects Database sheet from the xlsx file
df = pd.read_excel('data/raw/IEA CCUS Projects Database 2025.xlsx', sheet_name='CCUS Projects Database')

# Filter for Europe region only
df = df[df['Region'] == 'Europe']

# Remove all projects with status "Cancelled"
df = df[df['Project Status'] != 'Cancelled']

# Display basic information about the imported data
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

Shape: (357, 31)

Columns: ['Project name', 'ID', 'Country or economy', 'Partners', 'Project type', 'Announcement', 'FID', 'Operation', 'Suspension/decommissioning/cancellation', 'Project Status', 'Project phase', 'Announced capacity (Mt CO2/yr)', 'Estimated capacity by IEA (Mt CO2/yr)', 'Sector', 'Fate of carbon', 'Part of CCUS hub', 'Region', 'Ref 1', 'Ref 2', 'Ref 3', 'Ref 4', 'Ref 5', 'Ref 6', 'Ref 7', 'Link 1', 'Link 2', 'Link 3', 'Link 4', 'Link 5', 'Link 6', 'Link 7']


,Project name,ID,Country or economy,Partners,Project type,Announcement,FID,Operation,Suspension/decommissioning/cancellation,Project Status,...,Ref 5,Ref 6,Ref 7,Link 1,Link 2,Link 3,Link 4,Link 5,Link 6,Link 7
5,MOL Szank field CO2 EOR,669,Hungary,MOL group,Full chain,NaN,NaN,1992.0,NaN,Operational,...,NaN,NaN,NaN,Link 1,NaN,NaN,NaN,NaN,NaN,NaN
6,Sleipner,374,Norway,"Equinor, Eni",Full chain,1991.0,NaN,1996.0,NaN,Operational,...,NaN,NaN,NaN,Link 1,Link 2,NaN,NaN,NaN,NaN,NaN
7,Acorn CCS phase 2,9,United Kingdom,"Storegga(30%), Shell (30%), Harbour (30%), Nor...",T&S,2017.0,NaN,NaN,NaN,Planned,...,https://www.scottishconstructionnow.com/articl...,https://www.energyvoice.com/oilandgas/north-se...,https://www.theacornproject.uk/news-and-events...,Link 1,Link 2,Link 3,Link 4,Link 5,Link 6,Link 7
8,OCAP,1096,Netherlands,OCAP (Linde),Transport,2005.0,NaN,2005.0,NaN,Operational,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,Snohvit CO2 capture and storage,375,Norway,"Equinor, Petoro, TotalEnergies, Eni (following...",Full chain,2002.0,2002.0,2008.0,NaN,Operational,...,NaN,NaN,NaN,Link 1,Link 2,Link 3,NaN,NaN,NaN,NaN


In [18]:
import re

# Clean the Partners data for bipartite graph
# Extract project names and partners
partners_data = df[['Project name', 'Partners']].copy()

# Remove rows where Partners is NaN
partners_data = partners_data.dropna(subset=['Partners'])

# Split partners by comma and create edge list
edge_list = []
for idx, row in partners_data.iterrows():
    project = row['Project name']
    # Remove parentheses and their contents first
    partners_clean = re.sub(r'\([^)]*\)', '', str(row['Partners']))
    # Then split partners by comma and clean whitespace
    partners = [p.strip() for p in partners_clean.split(',')]
    # Create an edge for each partner
    for partner in partners:
        if partner:  # Skip empty strings
            edge_list.append({'Project': project, 'Partner': partner})

# Create DataFrame from edge list
bipartite_edges = pd.DataFrame(edge_list)

print(f"Total edges (project-partner pairs): {len(bipartite_edges)}")
print(f"Unique projects: {bipartite_edges['Project'].nunique()}")
print(f"Unique partners: {bipartite_edges['Partner'].nunique()}")
print(f"\nFirst 10 edges:")
bipartite_edges

Total edges (project-partner pairs): 823
Unique projects: 356
Unique partners: 391

First 10 edges:


,Project,Partner
0,MOL Szank field CO2 EOR,MOL group
1,Sleipner,Equinor
2,Sleipner,Eni
3,Acorn CCS phase 2,Storegga
4,Acorn CCS phase 2,Shell
...,...,...
818,GLOBE ( Lhoist Marche-les-Dames CCS),Lhoist
819,Lynemouth Power Station (BECCS),EP UK Investments
820,NL CCS Direct Injection Phase 2,Eni
821,NL CCS Direct Injection Phase 2,EBN


In [19]:
# Create node list of partners
partner_nodes = pd.DataFrame({
    'Partner': bipartite_edges['Partner'].unique()
})

# Sort alphabetically
partner_nodes = partner_nodes.sort_values('Partner').reset_index(drop=True)

print(f"Total unique partners: {len(partner_nodes)}")
print(f"\nFirst 20 partners:")
print(partner_nodes.head(20))

# Export to CSV
partner_nodes.to_csv('data/intermediate/partner_nodes.csv', index=False)
print(f"\n✓ Exported to 'data/intermediate/partner_nodes.csv'")

Total unique partners: 391

First 20 partners:
                               Partner
0                                    0
1                                  A2X
2                                ADNOC
3                        AEB Amsterdam
4                              ANDRITZ
5                                  ARC
6                                 ARGO
7                    Aalborg Forsyning
8                     Aalborg Portland
9                                Acorn
10                           Agglo Pau
11                              Aghata
12                         Air Liquide
13                  Air Liquide Polska
14                        Air Products
15                          AirLiquide
16                             Aker BP
17  Aker BP   PGNiG Upstream Norway AS
18                         Aker BP ASA
19                 Aker Carbon Capture

✓ Exported to 'data/intermediate/partner_nodes.csv'


In [20]:
# Create a mapping file for partner name standardization
# Format: Canonical_Name | Alias1 | Alias2 | Alias3 | ...
# Each row defines a canonical name and all its aliases

unique_partners = sorted(bipartite_edges['Partner'].unique())

partner_mapping = pd.DataFrame({
    'Canonical_Name': unique_partners,
    'Alias1': [''] * len(unique_partners),
    'Alias2': [''] * len(unique_partners),
    'Alias3': [''] * len(unique_partners),
    'Alias4': [''] * len(unique_partners),
    'Alias5': [''] * len(unique_partners),
    'Alias6': [''] * len(unique_partners),
    'Alias7': [''] * len(unique_partners)
})

# Export to CSV for manual editing
partner_mapping.to_csv('data/intermediate/partner_name_mapping.csv', index=False)

print(f"Total unique partners: {len(partner_mapping)}")
print(f"\n✓ Exported to 'data/intermediate/partner_name_mapping.csv'")
print("\nInstructions:")
print("- Column 1 (Canonical_Name): The final standardized name")
print("- Columns 2+ (Alias1, Alias2, ...): Other names that map to this canonical name")
print("- Add more alias columns if needed")
print(f"\nFirst 10 entries:")
partner_mapping.head(10)

Total unique partners: 391

✓ Exported to 'data/intermediate/partner_name_mapping.csv'

Instructions:
- Column 1 (Canonical_Name): The final standardized name
- Columns 2+ (Alias1, Alias2, ...): Other names that map to this canonical name
- Add more alias columns if needed

First 10 entries:


,Canonical_Name,Alias1,Alias2,Alias3,Alias4,Alias5,Alias6,Alias7
0,0,,,,,,,
1,A2X,,,,,,,
2,ADNOC,,,,,,,
3,AEB Amsterdam,,,,,,,
4,ANDRITZ,,,,,,,
5,ARC,,,,,,,
6,ARGO,,,,,,,
7,Aalborg Forsyning,,,,,,,
8,Aalborg Portland,,,,,,,
9,Acorn,,,,,,,


In [21]:
# After editing the mapping file, use this cell to apply the standardization
# Load the edited mapping
mapping = pd.read_csv('data/intermediate/partner_name_mapping_zoe.csv', encoding='latin-1')

# Build mapping dictionary from canonical name and all aliases
name_dict = {}
for _, row in mapping.iterrows():
    canonical = row['Canonical_Name']
    # Map the canonical name to itself
    name_dict[canonical] = canonical
    # Map each alias to the canonical name
    for col in mapping.columns[1:]:  # Skip first column (Canonical_Name)
        alias = row[col]
        if pd.notna(alias) and alias.strip():  # Only if alias is not empty
            name_dict[alias.strip()] = canonical

# Apply the mapping to create standardized edges
bipartite_edges_clean = bipartite_edges.copy()
bipartite_edges_clean['Partner'] = bipartite_edges_clean['Partner'].map(name_dict).fillna(bipartite_edges_clean['Partner'])

print(f"Before standardization: {bipartite_edges['Partner'].nunique()} unique partners")
print(f"After standardization: {bipartite_edges_clean['Partner'].nunique()} unique partners")
print(f"Merged {bipartite_edges['Partner'].nunique() - bipartite_edges_clean['Partner'].nunique()} duplicate partners")

# Export cleaned edge list
bipartite_edges_clean.to_csv('data/processed/bipartite_edges_clean.csv', index=False)
print(f"\n✓ Exported cleaned edges to 'data/processed/bipartite_edges_clean.csv'")

# Export updated partner node list
partner_nodes_clean = pd.DataFrame({
    'Partner': sorted(bipartite_edges_clean['Partner'].unique())
})
partner_nodes_clean.to_csv('data/processed/partner_nodes_clean.csv', index=False)
print(f"✓ Exported cleaned partner nodes to 'data/processed/partner_nodes_clean.csv'")

Before standardization: 391 unique partners
After standardization: 329 unique partners
Merged 62 duplicate partners

✓ Exported cleaned edges to 'data/processed/bipartite_edges_clean.csv'
✓ Exported cleaned partner nodes to 'data/processed/partner_nodes_clean.csv'
